# Test handling tilted tanks

In [ ]:
%load_ext autoreload
%autoreload 2
from source.cylinder_extraction.internal_station import Station
from pathlib import Path
from source.cylinder_alignment.internal_bundle_station import InternalBundleStation
import numpy as np
import pickle
import pickle
import sys
import json
sys.path.append('D:\Dev\GUI\processing_source_code')

In [ ]:
bundle = InternalBundleStation(data_dir='D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\top')

In [ ]:
dir = 'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\top\handle_tilted_tanks'
a = bundle.transform_and_write_final_icp_to_original_data(write_dir=dir,ref_scan=3)

#============================= OK ==========================

# TEST runing Fine Alignment only

In [1]:
import sys
sys.path.append('.source')
sys.path.append('D:\Dev\GUI\processing_source_code')
from source.internal_scans.without_floating_roof.project import STAGE, InternalNoFloatRoofProject

if __name__=="__main__":
    proj = InternalNoFloatRoofProject(
        data_dir='D:\REGISTRATION DATA\Test_S3\\test_fine_alignment\without_floating_roof',
        custom_params={
            'target_object': 'column', 
            'distance_to_wall_min': 0.2, 
            'icp': {'estimate_scanner_position': True, 'ref_scan':0, 'auto_pair': True},
            'roughly_aligned': True
            }
    )
    #
    proj.run(
        phase=STAGE.ICP,
        mode='single'
    )

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
FINE ALIGNMENT
STATION 11
STATION 12
STATION 13
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 11_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\11.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 12_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\12.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 13_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\13.pcd
Full registration ...
Full registration ...
----SOURCE 0----
===== source_id=0, target_id=1 =====
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.

NameError: name 'log_dir' is not defined

In [2]:
import sys
sys.path.append('.source')
sys.path.append('D:\Dev\GUI\processing_source_code')
from source.internal_scans.with_floating_roof.project import STAGE, InternalFloatRoofProject

if __name__=="__main__":
    proj = InternalFloatRoofProject(
        data_dir='D:\REGISTRATION DATA\Test_S3\\test_fine_alignment\with_floating_roof',
        custom_params={
            'top':    {'icp': {'estimate_scanner_position': True}, 'ref_scan':0, 'roughly_aligned': True},
            'bottom': {'icp': {'estimate_scanner_position': True}, 'ref_scan':0, 'roughly_aligned': True}
            }
    )
    # 
    run_parts = ['bottom','top']
    proj.run(
        phase=STAGE.ICP,
        mode='single',
        parts = run_parts
    )

FINE ALIGNMENT
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 11_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\with_floating_roof\bottom\rough_alignment\11.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 12_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\with_floating_roof\bottom\rough_alignment\12.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 13_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\with_floating_roof\bottom\rough_alignment\13.pcd
Full registration ...
Full registration ...
----SOURCE 0----
===== source_id=0, target_id=1 =====
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.8467, RMSE 0.0589
[Open3D DEBUG] Residual : 2.50e-04 (# of elements : 2559819)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.8469, RMSE 0.0587
[Open3D DEBUG] Residual : 2.38e-04 (# of elements 

NameError: name 'log_dir' is not defined

#============================= OK ==========================

In [8]:
sys.path.append('D:\Dev\GUI\processing_source_code')
import pickle
with open('D:\REGISTRATION DATA\Test_S3\\test_fine_alignment\with_floating_roof\\bottom\\rough_alignment\\bundle.pkl','rb') as f:
    bundle_rough_info = pickle.load(f)

In [ ]:
'D:\Dev\GUI\processing_source_code'

In [ ]:
import json
with open('D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\fine_alignment_info1.txt','r') as f:
    info = json.load(f)

In [ ]:
import numpy as np
import pickle

In [ ]:
def inverse_transform_matrix(rough_transform_info,fine_transform_info,sel_ref_scan=0):
    with open(fine_transform_info, 'rb') as f:
        info_fine = json.load(f) 
    with open(rough_transform_info, 'rb') as f:
        bundle_rough_info = pickle.load(f)
    m_fine = np.array(info_fine[str(Path(rough_transform_info).parent / f'{bundle_rough_info.stations[sel_ref_scan].id}.pcd')])
    m_rough = np.identity(4)
    m_rough[:3,:3] = bundle_rough_info.stations[sel_ref_scan].align['rotation']
    m_rough[:3,-1:] = bundle_rough_info.stations[sel_ref_scan].align['translation']
    inverse_matrix = np.linalg.inv(m_fine)@np.linalg.inv(m_rough)
    return inverse_matrix

In [ ]:
def inverse_transform_matrix(rough_transform_info,fine_transform_info,sel_ref_scan=0):
    with open(fine_transform_info, 'rb') as f:
        info_fine = json.load(f) 
    with open(rough_transform_info, 'rb') as f:
        bundle_rough_info = pickle.load(f)
    m_fine = np.array(info_fine[str(Path(rough_transform_info).parent / f'{bundle_rough_info.stations[sel_ref_scan].id}.pcd')])
    m_rough = np.identity(4)
    m_rough[:3,:3] = bundle_rough_info.stations[sel_ref_scan].align['rotation']
    m_rough[:3,-1:] = bundle_rough_info.stations[sel_ref_scan].align['translation']
    inverse_matrix = np.linalg.inv(m_fine) @ np.linalg.inv(m_rough)

    trans_matrices = []
    for key,value in info_fine.items(): #range(len(bundle_rough_info.stations)):
        #value = np.array(info_fine[str(Path(rough_transform_info).parent / f'{bundle_rough_info.stations[j].id}.pcd')])
        trans_matrices.append(inverse_matrix @ value) 
    
    return trans_matrices, inverse_matrix, bundle_rough_info, info_fine

In [ ]:
with open(fine_transform_info, 'rb') as f:
    info_fine = json.load(f) 

In [ ]:
bundle.stations

In [ ]:
import os
a = os.path.join('D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\rough_alignment','bundle.pkl')

In [ ]:
Path(a).parent

In [ ]:
str('D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\rough_alignment' / 'bundle.pkl')

In [ ]:
rough_transform_info = 'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\rough_alignment\\bundle.pkl'
fine_transform_info = 'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\fine_alignment_info1.txt'
#inverse_matrix = inverse_transform_matrix(rough_transform_info,fine_transform_info,sel_ref_scan=3)

In [ ]:
from source.cylinder_extraction.internal_station import Station
from pathlib import Path


In [ ]:
len(info_fine)

In [ ]:
trans_matrices,_,bundle,info_fine = inverse_transform_matrix(rough_transform_info,fine_transform_info,sel_ref_scan=3)
ind = 0
for key,value in info_fine.items():    
    s = Station(file_path=str(Path(rough_transform_info).parent / f'{Path(key).stem}.pcd'))
    s.transform(trans_matrices[ind][:3,:3],trans_matrices[ind][:3,-1:])
    s.write_points(filename=f'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\back2origin\FromRough\{Path(key).stem}.pcd',data_type='pcd')
    ind = ind + 1

In [ ]:
for i in range(len(bundle.stations)):
    s = Station(file_path=str(Path(fine_transform_info).parent / f'{bundle.stations[i].id}.pcd'))
    s.transform(inverse_matrix[:3,:3],inverse_matrix[:3,-1:])
    s.write_points(filename=f'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\back2origin\sel_ref_scan=3\{bundle.stations[i].id}.pcd',data_type='pcd')

In [ ]:
f'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\back2origin\{bundle.stations[1].id}.pcd'

In [ ]:
[str(Path(fine_transform_info).parent / f'{bundle.stations[0].id}.pcd')]

In [ ]:
for r in inverse_matrix:
    for c in r:
        print(c,end=' ')
    print('')

In [ ]:
0.999857692 -0.0154600751 -0.00675125926 8.74511082,
0.0155036283 0.999859027  0.00644675789,
        -1.20724550e+00],
       [ 6.65064016e-03, -6.55050947e-03,  9.99956426e-01,
        -3.85565689e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00

In [ ]:
with open('D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\rough_alignment\\bundle.pkl', 'rb') as f:
    bundle = pickle.load(f)

In [ ]:
bundle.stations[0].align

In [ ]:
0.99985769  0.01550363  0.00665064 -8.69950707
-0.01546008  0.99985903 -0.00655051 1.31701889
-0.00675126  0.00644676  0.99995643 3.92231223
0 0 0 1

In [ ]:
bundle.rough_align

In [ ]:
bundle.stations[0].shell

In [ ]:
bundle.stations[1].shell

In [ ]:
from source.cylinder_extraction.station import Station

In [ ]:
s = Station(file_path='D:\REGISTRATION DATA\Test_S3\JB-734/bottom/Ent 3504_11.e57')

In [ ]:
s.read_points()

In [ ]:
s.ptc

In [ ]:
points_2d = s.ptc[:,:2]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()
ax.scatter(*zip(*points_2d))
plt.show()

In [ ]:
import alphashape

In [ ]:
alpha_shape = alphashape.alphashape(points_2d, 2.0)
alpha_shape

In [ ]:

data_dir='D:\REGISTRATION DATA\Test_S3\JB-751_TK_6115\\test_fine_alignment'
# s = Station(file_path=file_path')
bundle = InternalBundleStation(data_dir=data_dir)

In [ ]:
rough_transform_info = 'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\rough_alignment\\bundle.pkl'
fine_transform_info = 'D:\REGISTRATION DATA\Test_S3\JB-712_EP tank 3008\\bottom\\fine_alignment\\fine_alignment_info1.txt'
inverse_matrix = inverse_transform_matrix(rough_transform_info,fine_transform_info,sel_ref_scan=3)

# Read E57 files

In [1]:
from source.cylinder_extraction.internal_station import Station
from pathlib import Path
from source.cylinder_alignment.internal_bundle_station import InternalBundleStation
import numpy as np
import pickle
import pickle
import sys
import json
sys.path.append('D:\Dev\GUI\processing_source_code')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:

data_dir='D:\REGISTRATION DATA\Test_S3\\test_fine_alignment\without_floating_roof'
# s = Station(file_path=file_path')
bundle = InternalBundleStation(data_dir=data_dir)
bundle.create_rough_alingment_folder_and_write_pcds()


D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 11_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\11.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 12_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\12.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 13_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\13.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 14_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\14.pcd
D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof 15_SPATIAL_SUBSAMPLED.e57
write to D:\REGISTRATION DATA\Test_S3\test_fine_alignment\without_floating_roof\rough_alignment\15.pcd


In [ ]:
for s in bundle.stations:
    s.read_points()
    s.write_points(
        data_type = 'pcd',
        filename = os.path.join(
            data_dir, 'rough_alignment',f'{s.name}.pcd'
        )
    )